# 12 - Mean-Field Variational Inference with TensorFlow Probability

In the previous notebook, we implemented Mean-Field Variational Bayes directly.
This notebook shows the same basic idea using **TensorFlow Probability (TFP)**.

We will:
1. generate synthetic regression data
2. build a Bayesian linear regression model
3. define a mean-field surrogate posterior
4. optimize the variational objective
5. compare learned parameters with the true values

## Package note

This notebook uses `tensorflow` and `tensorflow_probability`.
If they are not installed in your environment, install them first before running the notebook.

In [ ]:
# Uncomment if needed
# %pip install tensorflow tensorflow-probability

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_probability as tfp

tfd = tfp.distributions
tfb = tfp.bijectors

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)
tf.random.set_seed(42)

## Part 1: Synthetic data

As before, we use synthetic data so we know the true regression coefficients.

In [ ]:
n_samples = 200
n_features = 4
true_beta = np.array([1.5, -0.8, 0.3, 2.0], dtype=np.float32)
noise_scale = 0.5

X = np.random.normal(size=(n_samples, n_features)).astype(np.float32)
y = X @ true_beta + np.random.normal(scale=noise_scale, size=n_samples).astype(np.float32)

print('X shape:', X.shape)
print('y shape:', y.shape)
print('True beta:', true_beta)
print('True noise scale:', noise_scale)

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(X[:, 0], y, alpha=0.7, color='steelblue')
plt.xlabel('First feature: X[:, 0]')
plt.ylabel('Response y')
plt.title('Synthetic regression data')
plt.show()

## Part 2: Bayesian linear regression model

We place priors on the regression weights and the noise scale.
For simplicity:
- weights have standard normal priors
- noise scale is positive and also learned

In [ ]:
X_tf = tf.constant(X)
y_tf = tf.constant(y)

def target_log_prob_fn(beta, noise_scale_raw):
    noise_scale = tf.nn.softplus(noise_scale_raw) + 1e-4

    beta_prior = tf.reduce_sum(tfd.Normal(loc=0.0, scale=1.0).log_prob(beta))
    noise_prior = tfd.Normal(loc=0.0, scale=1.0).log_prob(noise_scale_raw)

    mean = tf.linalg.matvec(X_tf, beta)
    likelihood = tf.reduce_sum(tfd.Normal(loc=mean, scale=noise_scale).log_prob(y_tf))

    return beta_prior + noise_prior + likelihood

## Part 3: Mean-field surrogate posterior

A mean-field approximation assumes independence across latent variables.
Here we approximate the posterior over all unknowns with a diagonal Gaussian in unconstrained space.

In [ ]:
surrogate_posterior = tfp.experimental.vi.build_factored_surrogate_posterior(
    event_shape=[n_features, 1],
    dtype=[tf.float32, tf.float32]
)

surrogate_posterior

## Part 4: Optimize the variational objective

TFP provides `fit_surrogate_posterior`, which optimizes the variational objective for us.

In [ ]:
optimizer = tf.optimizers.Adam(learning_rate=0.05)

losses = tfp.vi.fit_surrogate_posterior(
    target_log_prob_fn=target_log_prob_fn,
    surrogate_posterior=surrogate_posterior,
    optimizer=optimizer,
    num_steps=300,
    sample_size=20
)

losses = losses.numpy()
print('Final variational loss:', losses[-1])

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses, color='navy', linewidth=2)
plt.xlabel('Step')
plt.ylabel('Variational loss')
plt.title('TensorFlow Probability optimization')
plt.show()

## Part 4.5: Compare ELBO and IWAE bounds



We optimize a variational objective based on ELBO. Here we estimate:

- ELBO estimate: $\mathbb{E}_q[\log w]$

- IWAE-$K$ estimate: $\mathbb{E}[\log \frac{1}{K}\sum_{k=1}^K w_k]$



where $\log w = \log p(\theta, y) - \log q(\theta)$. For larger $K$, IWAE is typically a tighter lower bound.


In [ ]:
def estimate_elbo_iwae(K=1, n_repeats=200):

    elbo_vals = []

    iwae_vals = []



    for _ in range(n_repeats):

        beta_s, noise_raw_s = surrogate_posterior.sample(K)



        log_p = target_log_prob_fn(beta_s, noise_raw_s)

        log_q = surrogate_posterior.log_prob((beta_s, noise_raw_s))

        log_w = log_p - log_q



        elbo_est = tf.reduce_mean(log_w)

        iwae_est = tf.reduce_logsumexp(log_w) - tf.math.log(tf.cast(K, tf.float32))



        elbo_vals.append(float(elbo_est.numpy()))

        iwae_vals.append(float(iwae_est.numpy()))



    return np.mean(elbo_vals), np.mean(iwae_vals)





Ks = [1, 5, 20, 100]

print('K | ELBO estimate | IWAE-K estimate')

print('-' * 38)

for K in Ks:

    elbo_mean, iwae_mean = estimate_elbo_iwae(K=K, n_repeats=120)

    print(f'{K:>3} | {elbo_mean:>13.4f} | {iwae_mean:>15.4f}')


## Part 5: Inspect the learned posterior

Now we sample from the surrogate posterior and summarize the learned parameters.

In [ ]:
posterior_samples = surrogate_posterior.sample(2000)
beta_samples = posterior_samples[0].numpy()
noise_raw_samples = posterior_samples[1].numpy().reshape(-1)
noise_scale_samples = tf.nn.softplus(noise_raw_samples).numpy() + 1e-4

beta_mean = beta_samples.mean(axis=0)
noise_scale_mean = noise_scale_samples.mean()

print('True beta:          ', true_beta)
print('Posterior mean beta:', np.round(beta_mean, 4))
print()
print('True noise scale:   ', round(float(noise_scale), 4))
print('Posterior mean scale:', round(float(noise_scale_mean), 4))

In [ ]:
indices = np.arange(n_features)
width = 0.38

plt.figure(figsize=(8, 4))
plt.bar(indices - width / 2, true_beta, width=width, label='True beta', color='steelblue', edgecolor='black')
plt.bar(indices + width / 2, beta_mean, width=width, label='Posterior mean beta', color='darkorange', edgecolor='black')
plt.xticks(indices, [f'beta_{i}' for i in range(n_features)])
plt.ylabel('Coefficient value')
plt.title('True vs learned coefficients')
plt.legend()
plt.show()

## Summary

This notebook shows how the same mean-field variational idea can be implemented using TensorFlow Probability instead of writing the optimization loop by hand.

Main idea:
1. define a probabilistic model
2. define a factorized surrogate posterior
3. optimize the variational objective
4. inspect posterior samples and parameter estimates

In [ ]:
# Optional exercises
# 1) Change the number of samples and compare parameter recovery.
# 2) Change the optimizer learning rate and inspect the loss curve.
# 3) Increase the number of features and see how the approximation behaves.

pass